In [1]:
import neptune
import pandas as pd

# Edit run meta info

In [36]:
# Reopen by run ID (from sys/id)
run = neptune.init_run(
    with_id="DNAT-274",       # e.g. your run id
    project="amar-mesic/dna-thesis",
    mode="sync",         # important, otherwise it's read-only
    api_token="eyJhcGlfYWRkcmVzcyI6Imh0dHBzOi8vYXBwLm5lcHR1bmUuYWkiLCJhcGlfdXJsIjoiaHR0cHM6Ly9hcHAubmVwdHVuZS5haSIsImFwaV9rZXkiOiJkOTQ1Njc4MC0yOTcyLTRlMmQtYTMwMy0xOGYxZTAwMmIzZGUifQ==",
)

# Add or overwrite metadata
run["meta/experiment"] = "Synth_data-Genotype_aware-SPS"
run["meta/model"] = "Amar-DNANet_Lite"
run["meta/dataset"] = "proved_it"
run["meta/seed"] = 43
run["meta/fold"] = 1

# Add tags too (handy in UI)
run["sys/tags"].add(["exp:Synth_data-Genotype_aware-SPS", "model:Amar-DNANet_Lite", "dataset:proved_it", "seed:43", "fold:1"])

# Always stop after edits
run.stop()

[neptune] [info   ] Neptune initialized. Open in the app: https://app.neptune.ai/amar-mesic/dna-thesis/e/DNAT-274
[neptune] [info   ] Shutting down background jobs, please wait a moment...
[neptune] [info   ] Done!
[neptune] [info   ] Explore the metadata in the Neptune app: https://app.neptune.ai/amar-mesic/dna-thesis/e/DNAT-274/metadata


In [40]:
# connect to your Neptune project
project = neptune.init_project(
    "amar-mesic/dna-thesis",
    api_token="eyJhcGlfYWRkcmVzcyI6Imh0dHBzOi8vYXBwLm5lcHR1bmUuYWkiLCJhcGlfdXJsIjoiaHR0cHM6Ly9hcHAubmVwdHVuZS5haSIsImFwaV9rZXkiOiJkOTQ1Njc4MC0yOTcyLTRlMmQtYTMwMy0xOGYxZTAwMmIzZGUifQ==",
)

# fetch runs (maybe filtered by tag or name)
runs_table = project.fetch_runs_table(columns=[
    "sys/name", 
    "meta/experiment", "meta/model", "meta/dataset", "meta/seed", "meta/fold",
    "test/pixel_f1", "test/pixel_precision", "test/pixel_recall",
    "test/allele_f1", "test/allele_precision", "test/allele_recall"
])

[neptune] [info   ] Neptune initialized. Open in the app: https://app.neptune.ai/amar-mesic/dna-thesis/


In [41]:
# get dataframe
df = runs_table.to_pandas()

# rename to cleaner columns
df = df.rename(columns={
    "meta/experiment": "experiment",
    "meta/model": "model",
    "meta/dataset": "dataset",
    "meta/seed": "seed",
    "meta/fold": "fold",
})

df

,sys/creation_time,sys/id,sys/name,dataset,experiment,fold,model,seed,test/allele_f1,test/allele_precision,test/allele_recall,test/pixel_f1,test/pixel_precision,test/pixel_recall
0,2025-09-02 10:35:36.705,DNAT-275,DNANet-335+长-1正真:_正假-预：缩放-第2,proved_it,DNANet_Baseline,3.0,DNANet_Pretrained,NaN,0.7723,0.9119,0.6697,0.7936,0.8689,0.7304
1,2025-09-02 10:29:17.291,DNAT-274,670+长-1正真:0正假-无验证-考试[30..34]-预：缩放-种43,proved_it,Synth_data-Genotype_aware-SPS,1.0,Amar-DNANet_Lite,43.0,0.8234,0.9090,0.7526,0.8096,0.7689,0.8547
2,2025-09-02 10:28:08.857,DNAT-273,DNANet-134+长-1正真:_正假-预：缩放-第2,proved_it,DNANet_Baseline,2.0,DNANet_Pretrained,NaN,0.7851,0.9015,0.6953,0.7950,0.8554,0.7426
3,2025-09-02 10:16:36.397,DNAT-272,670+长-1正真:0正假-无验证-考试[40..43]-预：缩放-种43,proved_it,Synth_data-Genotype_aware-SPS,2.0,Amar-DNANet_Lite,43.0,0.8079,0.8749,0.7504,0.7885,0.7490,0.8325
4,2025-09-02 09:46:05.711,DNAT-271,670+长-1正真:0正假-无验证-考试[40..43]-预：缩放-种42,proved_it,Synth_data-Genotype_aware-SPS,2.0,Amar-DNANet_Lite,42.0,0.7490,0.7155,0.7858,0.4097,0.2600,0.9654
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
76,2025-07-02 08:10:54.198,DNAT-63,最好U网-100真:0假-早停-lr0.01-用exp调度0.98-第1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
77,2025-07-01 18:04:27.351,DNAT-62,最好U网-100真:0假-早停-lr0.01-用cyc调度-第1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
78,2025-07-01 17:07:54.342,DNAT-60,最好U网-100真:0假-早停-lr0.01-用cos调度-第1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
79,2025-07-01 14:13:53.630,DNAT-54,最好U网-100真:0假-早停-lr0.01-不调度-第1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [43]:
# melt to long format (so you can aggregate all test uniformly)
df_long = df.melt(
    id_vars=["experiment", "model", "dataset", "seed", "fold"],
    value_vars=[
        "test/pixel_f1", "test/pixel_precision", "test/pixel_recall",
        "test/allele_f1", "test/allele_precision", "test/allele_recall"
    ],
    var_name="metric",
    value_name="value"
)
df_long

,experiment,model,dataset,seed,fold,metric,value
0,DNANet_Baseline,DNANet_Pretrained,proved_it,NaN,3.0,test/pixel_f1,0.7936
1,Synth_data-Genotype_aware-SPS,Amar-DNANet_Lite,proved_it,43.0,1.0,test/pixel_f1,0.8096
2,DNANet_Baseline,DNANet_Pretrained,proved_it,NaN,2.0,test/pixel_f1,0.7950
3,Synth_data-Genotype_aware-SPS,Amar-DNANet_Lite,proved_it,43.0,2.0,test/pixel_f1,0.7885
4,Synth_data-Genotype_aware-SPS,Amar-DNANet_Lite,proved_it,42.0,2.0,test/pixel_f1,0.4097
...,...,...,...,...,...,...,...
481,NaN,NaN,NaN,NaN,NaN,test/allele_recall,NaN
482,NaN,NaN,NaN,NaN,NaN,test/allele_recall,NaN
483,NaN,NaN,NaN,NaN,NaN,test/allele_recall,NaN
484,NaN,NaN,NaN,NaN,NaN,test/allele_recall,NaN


In [46]:
# group + aggregate
summary = (
    df_long
    .groupby(["experiment", "model", "dataset", "metric"])
    .agg(mean=("value", "mean"), std=("value", "std"), 
         min=("value", "min"), max=("value", "max"), n=("value", "count"))
    .reset_index()
)

summary

,experiment,model,dataset,metric,mean,std,min,max,n
0,AT_Baseline,AT_75,proved_it,test/allele_f1,0.758167,0.004325,0.75360,0.76220,3
1,AT_Baseline,AT_75,proved_it,test/allele_precision,0.799100,0.010400,0.78710,0.80550,3
2,AT_Baseline,AT_75,proved_it,test/allele_recall,0.721433,0.012287,0.70810,0.73230,3
3,AT_Baseline,AT_75,proved_it,test/pixel_f1,0.148100,0.002835,0.14490,0.15030,3
4,AT_Baseline,AT_75,proved_it,test/pixel_precision,0.080637,0.001725,0.07868,0.08194,3
5,AT_Baseline,AT_75,proved_it,test/pixel_recall,0.906800,0.006391,0.89960,0.91180,3
6,DNANet_Baseline,DNANet_Pretrained,proved_it,test/allele_f1,0.792000,0.023909,0.77230,0.81860,3
7,DNANet_Baseline,DNANet_Pretrained,proved_it,test/allele_precision,0.918100,0.020419,0.90150,0.94090,3
8,DNANet_Baseline,DNANet_Pretrained,proved_it,test/allele_recall,0.696500,0.027420,0.66970,0.72450,3
9,DNANet_Baseline,DNANet_Pretrained,proved_it,test/pixel_f1,0.820933,0.046136,0.79360,0.87420,3
